# NB06 - Sharpness-Aware Minimization (SAM)
## Background
SAM (Foret et al., 2021) explicitly seeks flat minima by perturbing model weights in the direction that maximally increases the loss, then taking a gradient step from the perturbed point. This two-step process targets the minimax objective: min_w max_{||e||<=rho} L(w+e). SAM significantly improves generalization across vision, NLP, and scientific ML tasks, often outperforming increased regularization or data augmentation alone. ASAM (Kwon et al., 2021) adapts the perturbation to parameter scale, and mSAM (Wen et al., 2022) adds micro-batch sharpness control.

## What You'll Learn
- SAM: two-step update (ascent then descent) from scratch
- ASAM: adaptive normalization of the perturbation ball
- Sharpness trajectory during training
- Generalization improvement on an overfitting benchmark

## Why This Matters
SAM has shown 1-2% top-1 ImageNet improvement over AdamW baselines with the same compute, and is used in state-of-the-art vision models. For language models, it helps on tasks where generalization matters more than memorization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple

# ── SAM: Sharpness-Aware Minimization ─────────────────────────────────────
@dataclass
class SAM:
    """SAM wrapper around a base optimizer."""
    rho: float = 0.05  # perturbation ball radius
    lr: float = 0.01
    momentum: float = 0.9
    _v: Optional[np.ndarray] = field(default=None, init=False)

    def _sgd_step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        """Inner SGD+Momentum for the actual weight update."""
        if self._v is None:
            self._v = np.zeros_like(params)
        self._v = self.momentum * self._v + grads
        return params - self.lr * self._v

    def step(self, params: np.ndarray, fn: Callable) -> Tuple[np.ndarray, float, float]:
        """SAM two-step: ascent then descent."""
        # Step 1: compute gradient at current params
        loss_before, grad = fn(params)
        # Step 2: compute e_hat = rho * grad / ||grad|| (perturbation)
        grad_norm = np.linalg.norm(grad)
        e_hat = self.rho * grad / (grad_norm + 1e-12)
        params_perturbed = params + e_hat
        # Step 3: compute gradient at perturbed point
        _, grad_perturbed = fn(params_perturbed)
        # Step 4: step in direction of perturbed gradient from original params
        params_new = self._sgd_step(params, grad_perturbed)
        return params_new, loss_before, grad_norm

    def reset(self):
        self._v = None

# ── Baseline SGD+Momentum for comparison ──────────────────────────────────
@dataclass
class SGD:
    lr: float = 0.01
    momentum: float = 0.9
    _v: Optional[np.ndarray] = field(default=None, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self._v is None: self._v = np.zeros_like(params)
        self._v = self.momentum * self._v + grads
        return params - self.lr * self._v

    def reset(self): self._v = None

print('SAM and baseline SGD implemented.')
print()
print('SAM update rule:')
print('  1. e_hat = rho * grad / ||grad||   (max perturbation direction)')
print('  2. w_tilde = w + e_hat              (perturbed weights)')
print('  3. g_tilde = grad_L(w_tilde)        (gradient at perturbed point)')
print('  4. w_new = SGD_step(w, g_tilde)     (update from original point)')
print()
print('This minimizes max_{||e||<=rho} L(w+e) instead of L(w) directly.')

In [ ]:
# ── Polynomial regression overfitting benchmark ───────────────────────────
np.random.seed(42)
n_train, n_test = 15, 200
x_train = np.linspace(-1, 1, n_train)
x_test  = np.linspace(-1, 1, n_test)
y_train = np.sin(2*np.pi*x_train) + 0.3*np.random.randn(n_train)
y_test  = np.sin(2*np.pi*x_test)

degree = 9
X_train = np.stack([x_train**i for i in range(degree+1)], axis=-1)
X_test  = np.stack([x_test**i for i in range(degree+1)], axis=-1)

def make_loss_fn(X, y):
    def loss_fn(w):
        pred = X @ w
        err = pred - y
        loss = np.mean(err**2)
        grad = 2 * X.T @ err / len(y)
        return loss, grad
    return loss_fn

train_fn = make_loss_fn(X_train, y_train)
test_fn  = make_loss_fn(X_test, y_test)

def run_sgd(n_steps=3000):
    opt = SGD(lr=0.02, momentum=0.9)
    opt.reset()
    w = np.zeros(degree+1)
    train_losses, test_losses, sharpnesses = [], [], []
    for _ in range(n_steps):
        loss, grad = train_fn(w)
        w = opt.step(w, grad)
        train_losses.append(loss)
        test_losses.append(test_fn(w)[0])
    return w, train_losses, test_losses

def run_sam(rho=0.1, n_steps=3000):
    opt = SAM(rho=rho, lr=0.02, momentum=0.9)
    opt.reset()
    w = np.zeros(degree+1)
    train_losses, test_losses = [], []
    for _ in range(n_steps):
        w, loss, grad_norm = opt.step(w, train_fn)
        train_losses.append(loss)
        test_losses.append(test_fn(w)[0])
    return w, train_losses, test_losses

w_sgd, sgd_train, sgd_test = run_sgd()
w_sam, sam_train, sam_test = run_sam(rho=0.1)

final_test_sgd = sgd_test[-1]
final_test_sam = sam_test[-1]
print(f'SGD final test MSE:  {final_test_sgd:.4f}')
print(f'SAM final test MSE:  {final_test_sam:.4f}')
print(f'Improvement:         {(final_test_sgd - final_test_sam) / final_test_sgd * 100:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training curves
axes[0].semilogy(sgd_train, label='SGD train', alpha=0.7)
axes[0].semilogy(sam_train, label='SAM train', alpha=0.7)
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Step')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(sgd_test, label='SGD test', alpha=0.7)
axes[1].semilogy(sam_test, label='SAM test', alpha=0.7)
axes[1].set_title('Test Loss (Generalization)'); axes[1].set_xlabel('Step')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Final predictions
axes[2].plot(x_test, y_test, 'k--', linewidth=2, label='True')
axes[2].plot(x_test, X_test @ w_sgd, 'b-', alpha=0.8, label=f'SGD (MSE={final_test_sgd:.3f})')
axes[2].plot(x_test, X_test @ w_sam, 'r-', alpha=0.8, label=f'SAM (MSE={final_test_sam:.3f})')
axes[2].scatter(x_train, y_train, c='k', s=20, zorder=5, label='Train data')
axes[2].set_title('Fitted Functions'); axes[2].set_ylim(-2, 2)
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{BASE}/s30_06_sam.png', dpi=100, bbox_inches='tight')
plt.show()
print('SAM vs SGD comparison complete.')